In [36]:
import re
import unicodedata
import pandas as pd
from pathlib import Path


processed_books = pd.read_csv(r"C:\Users\LOQ\project_books\.venv\datas\processing_data\processed_books_dataset_enriched.csv")
awards_book = pd.read_csv(r"C:\Users\LOQ\project_books\.venv\datas\books_by_award.csv")
FOLDER = Path(r"C:\Users\LOQ\project_books\.venv\datas\data_collection\bestsellers")
bestseller_file = list(FOLDER.glob("*.csv"))

bestsellers_df = []

for files in bestseller_file:
    df = pd.read_csv(files)
    df["bestseller_source"] = files.stem
    bestsellers_df.append(df)

bestseller_df = pd.concat(bestsellers_df, ignore_index=True)

print("total records :", len(bestseller_df))
print(bestseller_df.head())

total records : 299
                                       Book                 Author(s)  \
0                      A Tale of Two Cities           Charles Dickens   
1     The Little Prince ( Le Petit Prince )  Antoine de Saint-Exupéry   
2            The Alchemist ( O Alquimista )              Paulo Coelho   
3  Harry Potter and the Philosopher's Stone             J. K. Rowling   
4                  And Then There Were None           Agatha Christie   

  Original language First published                 Approximate sales  \
0           English            1859               >200 million [ 20 ]   
1            French            1943  200 million [ 21 ] [ 22 ] [ 23 ]   
2        Portuguese            1988         150 million [ 24 ] [ 25 ]   
3           English            1997         120 million [ 26 ] [ 27 ]   
4           English            1939                100 million [ 28 ]   

                Genre bestseller_source Approx. sales Book series  \
0  Historical fiction     books_t

In [37]:
def clean_author(author):
    if pd.isna(author):
        return ""
    author = unicodedata.normalize("NFKC", str(author))
    author = author.lower()
    author = re.sub(r"[^\w\s,]", "", author, flags=re.UNICODE)
    author = re.sub(r"\s+", " ", author).strip()
    author = re.sub(r"\s*,\s*", ", ", author)
    return author

def normalize_title(title):
    title = str(title)
    title = unicodedata.normalize("NFKC", title)
    title = title.lower()
    title = re.sub(r"\([^)]*\)", "", title)
    title = re.sub(r"[^\w\s]", " ", title, flags=re.UNICODE)
    title = re.sub(r"\s+", " ", title).strip()
    return title

In [38]:
awards_book["title_clean"] = awards_book["book"].apply(normalize_title)
bestseller_df["title_clean"] = bestseller_df["Book"].apply(normalize_title)
bestseller_df["author_clean"] = bestseller_df["Author(s)"].apply(clean_author)

In [39]:
bestseller_df["author_clean"].head(20)

0                          charles dickens
1                  antoine de saintexupéry
2                             paulo coelho
3                              j k rowling
4                          agatha christie
5                               cao xueqin
6                            j r r tolkien
7                            lewis carroll
8                chief editor wei jiangong
9                       robert badenpowell
10                 william holmes mcguffey
11                         various authors
12              kazuko hosoki kaori hosoki
13                            noah webster
14                         various authors
15                     general mills staff
16     choi park lais family, among others
17                          merriamwebster
18                        peter mark roget
19    impress dekiru series editorial desk
Name: author_clean, dtype: str

In [40]:
# Create a mapping from book title to award name(s)
awards_by_title = (
    awards_book.dropna(subset=["title_clean", "award"])
    .groupby("title_clean")["award"]
    .agg(lambda names: sorted(set(names)))
    .to_dict()
)

# Mark award-winning books and specify their awards
processed_books["is_award_winner"] = (
    processed_books["title_clean"].isin(awards_by_title)
)

processed_books["award_names"] = (
    processed_books["title_clean"].map(awards_by_title)
)

# Books with no matching award get an empty list
processed_books["award_names"] = (
    processed_books["award_names"].apply(
        lambda x: x if isinstance(x, list) else []
    )
)

print(
    processed_books[
        ["title_clean", "is_award_winner", "award_names"]
    ].head(20)
)
print("total award winning books:", len(processed_books[processed_books["is_award_winner"]==True]))

                                          title_clean  is_award_winner  \
0                                 the face of another            False   
1                               generations of winter            False   
2                            the street of crocodiles            False   
3                                              kokoro            False   
4           sense and sensibility illustrated edition            False   
5                                   the final problem            False   
6                            a house boat on the styx            False   
7                                         virgin soil            False   
8                                           hemingway            False   
9                                              hunger            False   
10                            italian science fiction            False   
11                                            oblomov            False   
12        principles of phisics or nat

In [41]:
processed_books[processed_books["is_award_winner"]==True]

,google_id,title,subtitle,authors,publisher,published_date,description,categories,page_count,language,...,categories_sbert,description_final,author_clean,title_clean,category_clean,clean_description,author_set,completeness,is_award_winner,award_names
59,1lI_PgAACAAJ,The God of Small Things,NaN,Arundhati Roy,NaN,1999,Revised traditional Chinese edition of The God...,NaN,312.0,en,...,"Literary Fiction, Women's Fiction, dalit, Cath...",Revised traditional Chinese edition of The God...,arundhati roy,the god of small things,"literary fiction, women s fiction, dalit, cath...",Revised traditional Chinese edition of The God...,{'arundhati roy'},5,True,[Booker Prize–winning works]
86,zjRQAAAAYAAJ,Northern Lights,NaN,Philip Pullman,NaN,1995,"Accompanied by her daemon, Lyra Belacqua sets ...",Juvenile Fiction,420.0,en,...,"Juvenile Fiction, Political theology, Polar be...","Accompanied by her daemon, Lyra Belacqua sets ...",philip pullman,northern lights,"juvenile fiction, political theology, polar be...","Accompanied by her daemon, Lyra Belacqua sets ...",{'philip pullman'},4,True,[British Book Award–winning works]
232,qEKp0AEACAAJ,The Promise,NaN,Danielle Steel,NaN,1978,NaN,Large type books,0.0,en,...,"Large type books, Fiction in English, American...",Determined to get married despite parental dis...,danielle steel,the promise,"large type books, fiction in english, american...",Determined to get married despite parental dis...,{'danielle steel'},3,True,[Booker Prize–winning works]
355,A-AS46wG0p8C,The Da Vinci Code,NaN,Dan Brown,Random House,2004,Harvard professor Robert Langdon receives an u...,Fiction,484.0,en,...,"Suspense & Thriller, Spiritual Fiction, hieros...",Harvard professor Robert Langdon receives an u...,dan brown,the da vinci code,"suspense thriller, spiritual fiction, hieros g...",Harvard professor Robert Langdon receives an u...,{'dan brown'},7,True,[British Book Award–winning works]
559,j-6LMQEACAAJ,The Ocean at the End of the Lane,NaN,Neil Gaiman,Headline Review,2013,THE OCEAN AT THE END OF THE LANE is a novel ab...,Fiction,248.0,en,...,"Good and evil, Boys, Women, Magic, Memory, Sur...",THE OCEAN AT THE END OF THE LANE is a novel ab...,neil gaiman,the ocean at the end of the lane,"good and evil, boys, women, magic, memory, sur...",THE OCEAN AT THE END OF THE LANE is a novel ab...,{'neil gaiman'},7,True,[British Book Award–winning works]
1052,OWIauAEACAAJ,Harry Potter and the Half-Blood Prince,NaN,J. K. Rowling,Arthur a Levine,2005,Sixth-year Hogwarts student Harry Potter gains...,Juvenile Fiction,652.0,en,...,"Juvenile Fiction, orphans, foster homes, roman...",Sixth-year Hogwarts student Harry Potter gains...,j k rowling,harry potter and the half blood prince,"juvenile fiction, orphans, foster homes, roman...",Sixth-year Hogwarts student Harry Potter gains...,{'jk rowling'},7,True,[British Book Award–winning works]
1299,ZMLkjwEACAAJ,A Brief History of Seven Killings,NaN,Marlon James,Clipper Audio,2015-01-10,"On 3 December 1976, just weeks before Bob Marl...",Assassins,NaN,en,...,"Assassins, Assassination attempts, Reggae musi...","On 3 December 1976, just weeks before Bob Marl...",marlon james,a brief history of seven killings,"assassins, assassination attempts, reggae musi...","On 3 December 1976, just weeks before Bob Marl...",{'marlon james'},7,True,[Booker Prize–winning works]
1726,6RS00wtlWhIC,Life of Pi,A Novel,Yann Martel,Canongate Books,2001,"After the sinking of a crago ship, a solitary ...",Action and adventure fiction,337.0,en,...,Action and adventure fiction,"After the sinking of a crago ship, a solitary ...",yann martel,life of pi,action and adventure fiction,"After the sinking of a crago ship, a solitary ...",{'yann martel'},7,True,[Booker Prize–winning works]
1744,zWPAGgAACAAJ,The Amber Spyglass,NaN,Philip Pullman,NaN,2007,"""We're going to the land of the dead and we're...",Adventure,522.0,en,...,Adventure,"""We're going to the land of the dead and we're...",philip pullman,the amber spyglass,adventure

In [42]:
# IDs of books incorrectly marked as award winners
incorrect_ids = [
    "Y12-OQ5BDjcC",
    "InhwswEACAAJ",
    "QZWhtAEACAAJ"
]

# Correct their flags
processed_books.loc[
    processed_books["google_id"].isin(incorrect_ids),
    ["is_award_winner", "award_name"]
] = [False, None]

In [43]:
from rapidfuzz import fuzz
# Create a unique list of bestseller title-author pairs
bestseller_keys = (
    bestseller_df[["title_clean", "author_clean"]]
    .drop_duplicates()
    .dropna(subset=["title_clean"])
)

# Prepare columns in the main dataset
processed_books["is_bestseller"] = False
processed_books["bestseller_match_method"] = None

# Group bestseller records by title for faster candidate lookup
bestsellers_by_title = {
    title: group
    for title, group in bestseller_keys.groupby("title_clean")
}

for idx, book in processed_books.iterrows():

    title = book["title_clean"]
    author = book["author_clean"]

    # Find bestseller records with the same normalized title
    candidates = bestsellers_by_title.get(title)

    if candidates is None:
        continue

    # Compare this book's author against each candidate author
    best_score = 0

    for _, candidate in candidates.iterrows():
        candidate_author = candidate["author_clean"]

        if pd.isna(author) or pd.isna(candidate_author):
            continue

        score = fuzz.ratio(
            str(author),
            str(candidate_author)
        )

        best_score = max(best_score, score)

    # Use a conservative starting threshold
    if best_score >= 90:
        processed_books.at[idx, "is_bestseller"] = True
        processed_books.at[
            idx, "bestseller_match_method"
        ] = "exact_title_fuzzy_author"

In [44]:
processed_books[processed_books["is_bestseller"]==True].head()

,google_id,title,subtitle,authors,publisher,published_date,description,categories,page_count,language,...,title_clean,category_clean,clean_description,author_set,completeness,is_award_winner,award_names,award_name,is_bestseller,bestseller_match_method
26,2rxfPwAACAAJ,And Then There Were None,NaN,Agatha Christie,Paw Prints,2008-09-18,A killer stalks a group of ten total strangers...,Devon (England),0.0,en,...,and then there were none,"devon england , roman policier, french languag...",A killer stalks a group of ten total strangers...,{'agatha christie'},7,False,[],NaN,True,exact_title_fuzzy_author
127,gddq0QEACAAJ,The Hobbit,NaN,J. R. Tolkien,HMH Books For Young Readers,1984-10,J.R.R. Tolkien's classic is splendidly illustr...,Fiction,0.0,en,...,the hobbit,"fantasy, arkenstone, battle of five armies, in...",J.R.R. Tolkien's classic is splendidly illustr...,{'j r tolkien'},7,False,[],NaN,True,exact_title_fuzzy_author
129,202nQgAACAAJ,The Cat in the Hat,NaN,Dr. Seuss,HarperCollins,1958,Two children sitting at home on a rainy day ar...,Cats,61.0,en,...,the cat in the hat,"cats, nonsense verses, children s stories, ame...",Two children sitting at home on a rainy day ar...,{'seuss'},7,False,[],NaN,True,exact_title_fuzzy_author
244,EvqJCGeqKhsC,Pride and Prejudice,NaN,Jane Austen,"Oxford University Press, USA",1980,The text of Pride and Prejudice is the 1813 fi...,Fiction,388.0,en,...,pride and prejudice,"romance, historical, regency, british and iris...",The text of Pride and Prejudice is the 1813 fi...,{'jane austen'},7,False,[],NaN,True,exact_title_fuzzy_author
273,SxPUCwAAQBAJ,Me Before You (Movie Tie-In),A Novel,Jojo Moyes,Penguin Books,2016-04-26,Taking a job as an assistant to extreme sports...,Fiction,402.0,en,...,me before you,"young women, fiction / contemporary women, fic...",Taking a job as an assistant to extreme sports...,{'jojo moyes'},7,False,[],NaN,True,exact_title_fuzzy_author


In [45]:
import json
import sqlite3

DB_PATH = r"C:\Users\LOQ\project_books\.venv\datas\books.db"
processed_books_to_save = processed_books.copy()

list_columns = [
    col for col in processed_books.columns
    if processed_books[col].apply(
        lambda x: isinstance(x, list)
    ).any()
]
for col in list_columns:
    processed_books_to_save[col] = (
        processed_books_to_save[col].apply(
            lambda x: json.dumps(x, ensure_ascii=False)
            if isinstance(x, list)
            else x
        )
    )
conn = sqlite3.connect(DB_PATH)

processed_books_to_save.to_sql(
    "processed_books",
    conn,
    if_exists="replace",
    index=False
)

print("Books saved successfully!")

conn.close()

Books saved successfully!


In [46]:
processed_books.to_csv("processed_books_with_bestseller_award.csv", index=False)